# M9.13 · Görüntü kalite ölçütleri — MSE, PSNR, SSIM, MS-SSIM

Öznel ve nesnel kalite; tam referanslı, indirgenmiş referanslı ve referanssız ölçütler. MSE/PSNR'ın neden yanıltıcı olduğunu (aynı MSE'de çok farklı görünen görüntüler) ve SSIM'in parlaklık, kontrast, yapı ayrıştırmasını elle ve saf Python'da kuruyorsun; NumPy'da `skimage.metrics.structural_similarity` ile birebir SSIM ve Wang'ın "eşit MSE" figürünü üretiyorsun; PyTorch'ta türevlenebilir SSIM ile MSE küresi üzerinde SSIM'i en büyük ve en küçük yapan görüntüleri arıyorsun; en sonda toplu (batch) MS-SSIM'i vektörel yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-13-quality.html](https://m-gonul.github.io/pytorch-lab/m9-13-quality.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_13_quality.py
CHECKS = {
 "b": "import math\nimport numpy as np\nfrom skimage import data\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"skimage\", \"cv2\", \"scipy\"}))\n    rng = np.random.default_rng(0)\n    x = data.camera()[100:132, 100:132].astype(float).ravel()\n    y = np.clip(x + rng.normal(0, 20, x.shape), 0, 255)\n\n    def mp():\n        close(need(ns, \"mse\")(x.tolist(), y.tolist()), ((x - y) ** 2).mean(), what=\"MSE\")\n        close(need(ns, \"psnr\")(x.tolist(), y.tolist()), 10 * np.log10(255 ** 2 / ((x - y) ** 2).mean()), what=\"PSNR\")\n        ensure(ns[\"psnr\"]([1, 2], [1, 2]) == float(\"inf\"), \"özdeş → inf\")\n    R.test(\"mse, psnr\", mp)\n\n    def ss():\n        C1, C2 = 6.5025, 58.5225\n        mx, my = x.mean(), y.mean(); vx, vy = x.var(ddof=1), y.var(ddof=1); c = np.cov(x, y)[0, 1]\n        want = (2 * mx * my + C1) * (2 * c + C2) / ((mx ** 2 + my ** 2 + C1) * (vx + vy + C2))\n        close(need(ns, \"ssim_global\")(x.tolist(), y.tolist()), want, atol=1e-12, what=\"SSIM\")\n        l, cc, s = need(ns, \"ssim_components\")(x.tolist(), y.tolist())\n        close(l * cc * s, want, atol=1e-12, what=\"l·c·s = SSIM (C3 = C2/2)\")\n        close(ns[\"ssim_global\"](x.tolist(), x.tolist()), 1.0, what=\"özdeş → 1\")\n        close(ns[\"ssim_global\"](x.tolist(), y.tolist()), ns[\"ssim_global\"](y.tolist(), x.tolist()), what=\"simetri\")\n        return f\"l = {l:.4f}, c = {cc:.4f}, s = {s:.4f}\"\n    R.test(\"ssim_global ve bileşenleri\", ss)\n",
 "c": "import numpy as np\nfrom skimage import data\nfrom skimage.metrics import structural_similarity as sk_ssim\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    img = data.camera()\n\n    def eq():\n        rng = np.random.default_rng(0)\n        for y in (np.clip(img + rng.normal(0, 20, img.shape), 0, 255), img[::-1].astype(float), img * 0.5 + 40):\n            want = sk_ssim(img.astype(float), y.astype(float), data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)\n            close(need(ns, \"ssim\")(img, y), want, atol=1e-10, what=\"skimage SSIM\")\n    R.test(\"ssim = skimage structural_similarity\", eq)\n\n    def fig():\n        target = 210.0\n        rows = []\n        res = {}\n        for name in (\"mean_shift\", \"contrast\", \"noise\", \"impulse\", \"blur\", \"jpeg\"):\n            out = need(ns, name)(img, target)\n            m = float(((out - img.astype(float)) ** 2).mean())\n            tol = 0.15 if name == \"jpeg\" else 0.03\n            ensure(abs(m - target) / target < tol, f\"{name}: MSE {m:.1f}\")\n            res[name] = ns[\"ssim\"](img, out)\n            rows.append([name, f\"{m:.1f}\", f\"{res[name]:.3f}\"])\n        ensure(res[\"mean_shift\"] > 0.9 and res[\"mean_shift\"] == max(res.values()), \"ortalama kayması en yüksek SSIM'i almalı\")\n        ensure(res[\"blur\"] < 0.8 and res[\"jpeg\"] < 0.8 and res[\"noise\"] < 0.8, \"bulanıklık, JPEG ve gürültü düşük SSIM almalı\")\n        R.table(\"Aynı MSE (≈ 210), çok farklı kalite (Wang 2004, Şekil 2)\", [\"bozulma\", \"MSE\", \"SSIM\"], rows,\n                note=\"PSNR hepsine ≈ 24.9 dB der; gözünüz ve SSIM katılmaz.\")\n    R.test(\"eşit MSE figürü\", fig)\n",
 "d": "import numpy as np\nimport torch\nfrom skimage import data\nfrom skimage.metrics import structural_similarity as sk_ssim\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    ref = torch.tensor(data.camera()[128:384, 128:384], dtype=torch.float64)\n\n    def eq():\n        y = ref + torch.randn(ref.shape, generator=torch.Generator().manual_seed(0), dtype=torch.float64) * 20\n        want = sk_ssim(ref.numpy(), y.numpy(), data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)\n        close(float(need(ns, \"ssim_t\")(ref, y)), want, atol=1e-10, what=\"skimage SSIM\")\n        yy = y.clone().requires_grad_(True)\n        ns[\"ssim_t\"](ref, yy).backward()\n        ensure(yy.grad is not None and torch.isfinite(yy.grad).all(), \"türev\")\n    R.test(\"ssim_t = skimage ve türevlenebilir\", eq)\n\n    def sph():\n        hi = need(ns, \"mse_sphere\")(ref, 210.0, +1)\n        lo = ns[\"mse_sphere\"](ref, 210.0, -1)\n        m = lambda a: float(((a - ref) ** 2).mean())\n        ensure(abs(m(hi) - 210) < 0.5 and abs(m(lo) - 210) < 0.5, \"MSE küresinde kalmalı\")\n        sh, sl = float(ns[\"ssim_t\"](ref, hi)), float(ns[\"ssim_t\"](ref, lo))\n        ensure(sh > 0.9 and sl < 0.3, f\"en iyi {sh:.3f}, en kötü {sl:.3f}\")\n        R.table(\"MSE = 210 küresinde\", [\"görüntü\", \"MSE\", \"SSIM\"], [[\"SSIM'i en büyükle\", f\"{m(hi):.1f}\", f\"{sh:.3f}\"], [\"SSIM'i en küçükle\", f\"{m(lo):.1f}\", f\"{sl:.3f}\"]],\n                note=\"Aynı MSE: en iyisinde hata düz bölgelere değil yapının içine 'gizlenir', en kötüsünde yapıyı bozan desenler oluşur.\")\n    R.test(\"MSE küresinde en büyük / en küçük SSIM\", sph)\n",
 "e": "import numpy as np\nimport torch\nfrom scipy import ndimage\nfrom skimage import data, color\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\nW = (0.0448, 0.2856, 0.3001, 0.2363, 0.1333)\n\n\ndef ref_ms(x, y):\n    f = lambda a: ndimage.gaussian_filter(a, 1.5, truncate=3.5, mode=\"reflect\")\n    C1, C2 = 6.5025, 58.5225\n    out = 1.0\n    for j, w in enumerate(W):\n        ux, uy = f(x), f(y)\n        vx, vy, vxy = f(x * x) - ux * ux, f(y * y) - uy * uy, f(x * y) - ux * uy\n        cs = (2 * vxy + C2) / (vx + vy + C2)\n        if j < 4:\n            out *= max(cs[5:-5, 5:-5].mean(), 0) ** w\n            H, Wd = x.shape[0] // 2 * 2, x.shape[1] // 2 * 2\n            x = x[:H, :Wd].reshape(H // 2, 2, Wd // 2, 2).mean((1, 3)); y = y[:H, :Wd].reshape(H // 2, 2, Wd // 2, 2).mean((1, 3))\n        else:\n            l = (2 * ux * uy + C1) / (ux * ux + uy * uy + C1)\n            out *= max((l * cs)[5:-5, 5:-5].mean(), 0) ** w\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız ölçek döngüsü\", lambda: max_loops(src, 1, func=\"ms_ssim\"))\n    rng = np.random.default_rng(0)\n    ref = np.stack([data.camera()[:256, :256], color.rgb2gray(data.astronaut())[:256, :256] * 255, data.moon()[:256, :256], data.coins()[:256, :256]]).astype(np.float64)\n    dist = np.clip(ref + rng.normal(0, 25, ref.shape), 0, 255)\n\n    def eq():\n        got = need(ns, \"ms_ssim\")(torch.tensor(ref), torch.tensor(dist)).numpy()\n        want = np.array([ref_ms(a, b) for a, b in zip(ref, dist)])\n        close(got, want, atol=1e-10, what=\"görüntü başına başvuru\")\n        close(ns[\"ms_ssim\"](torch.tensor(ref), torch.tensor(ref)).numpy(), np.ones(4), atol=1e-12, what=\"özdeş → 1\")\n    R.test(\"ms_ssim = başvuru (4 görüntülük yığın)\", eq)\n    if \"ms_ssim\" not in ns:\n        return\n\n    def tab():\n        img = ref[0]\n        ss = lambda a: ns[\"ms_ssim\"](torch.tensor(img[None]), torch.tensor(a[None])).item()\n        import cv2\n        rows = [[\"Gauss gürültüsü σ = 25\", f\"{ss(np.clip(img + rng.normal(0, 25, img.shape), 0, 255)):.3f}\"],\n                [\"Gauss bulanıklık σ = 2\", f\"{ss(cv2.GaussianBlur(img, (0, 0), 2)):.3f}\"],\n                [\"ortalama +15\", f\"{ss(img + 15):.3f}\"]]\n        R.table(\"camera, MS-SSIM\", [\"bozulma\", \"MS-SSIM\"], rows)\n    R.test(\"MS-SSIM tablosu\", tab)\n    big_r, big_d = torch.tensor(np.tile(ref, (4, 1, 1))), torch.tensor(np.tile(dist, (4, 1, 1)))\n    t1 = bench(lambda: [ref_ms(a, b) for a, b in zip(big_r.numpy(), big_d.numpy())], repeat=1, number=1)\n    t2 = bench(lambda: ns[\"ms_ssim\"](big_r, big_d), repeat=3)\n    R.table(\"16 × 256²\", [\"yöntem\", \"süre\"], [[\"görüntü döngüsü (scipy)\", fmt_time(t1)], [\"toplu tensör\", fmt_time(t2)]])\n",
 "x1": "import numpy as np\nfrom skimage import data\nfrom skimage.metrics import structural_similarity as sk\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    rng = np.random.default_rng(0)\n    x = data.camera().astype(float)\n    y = np.clip(x + rng.normal(0, 15, x.shape), 0, 255)\n    rows = []\n    for win, sc in [(7, True), (7, False), (11, True), (3, False)]:\n        def t(win=win, sc=sc):\n            m, S = need(ns, \"ssim_map\")(x, y, win, sc)\n            wm, wS = sk(x, y, win_size=win, use_sample_covariance=sc, data_range=255, full=True)\n            close(m, wm, atol=1e-10, what=\"ortalama\"); close(S, wS, atol=1e-10, what=\"harita\")\n            rows.append([win, \"örnek\" if sc else \"popülasyon\", f\"{m:.4f}\"])\n        R.test(f\"ssim_map = skimage (pencere {win}, {'örnek' if sc else 'popülasyon'} kovaryansı)\", t)\n    if rows:\n        R.table(\"Aynı görüntü çifti, farklı SSIM tanımları\", [\"pencere\", \"kovaryans\", \"SSIM\"], rows, note=\"Makaleleri karşılaştırırken hangi SSIM tanımının kullanıldığına bakın: sayılar değişir.\")\n",
 "x2": "import numpy as np\nimport cv2\nfrom skimage import data, color\nfrom skimage.metrics import structural_similarity as sk\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    hr = data.astronaut()\n    lr = cv2.resize(hr, (128, 128), interpolation=cv2.INTER_AREA)\n    up = cv2.resize(lr, (512, 512), interpolation=cv2.INTER_CUBIC)\n\n    def y():\n        close(need(ns, \"rgb_to_y\")(hr), color.rgb2ycbcr(hr)[..., 0], atol=1e-9, what=\"skimage rgb2ycbcr Y\")\n    R.test(\"rgb_to_y = skimage.color.rgb2ycbcr\", y)\n\n    def ps():\n        p0, p4 = need(ns, \"psnr_y\")(hr, up), ns[\"psnr_y\"](hr, up, 4)\n        rgb = 10 * np.log10(255 ** 2 / ((hr.astype(float) - up) ** 2).mean())\n        s = need(ns, \"ssim_rgb\")(hr, up)\n        close(s, sk(hr.astype(float), up.astype(float), channel_axis=2, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False), atol=1e-10, what=\"kanal ortalamalı SSIM\")\n        R.table(\"4× bikübik büyütme (astronaut)\", [\"ölçüt\", \"değer\"], [[\"PSNR (RGB)\", f\"{rgb:.2f}\"], [\"PSNR (Y)\", f\"{p0:.2f}\"], [\"PSNR (Y, kenar 4 kırpık)\", f\"{p4:.2f}\"], [\"SSIM (RGB kanal ortalaması)\", f\"{s:.4f}\"]],\n                note=\"Aynı çıktı için dört farklı 'PSNR': makaleler arası karşılaştırmada protokolü mutlaka kontrol et.\")\n    R.test(\"psnr_y ve ssim_rgb\", ps)\n",
 "x3": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import need, ensure\n\n\ndef check(ns, R):\n    base = cv2.GaussianBlur(data.camera().astype(np.float64), (0, 0), 2)\n\n    def nz():\n        rows = []\n        for s in (5, 10, 20):\n            img = base + np.random.default_rng(s).normal(0, s, base.shape)\n            e = need(ns, \"estimate_noise\")(img)\n            ensure(abs(e - s) / s < 0.15, f\"σ = {s}: kestirim {e:.2f}\")\n            rows.append([s, f\"{e:.2f}\"])\n        R.table(\"Immerkær kestirimi (yumuşak görüntü + Gauss gürültüsü)\", [\"gerçek σ\", \"kestirim\"], rows, note=\"Dokulu görüntülerde kenarlar kestirimi şişirir; bu yüzden yöntemler genelde düz bölgeleri seçer.\")\n    R.test(\"estimate_noise\", nz)\n\n    def bl():\n        img = data.camera()\n        vals = [need(ns, \"blur_score\")(cv2.GaussianBlur(img, (0, 0), s) if s else img) for s in (0, 1, 2, 4)]\n        ensure(all(a > b for a, b in zip(vals, vals[1:])), f\"bulanıklaştıkça azalmalı: {vals}\")\n        return \" → \".join(f\"{v:.0f}\" for v in vals) + \" (σ = 0, 1, 2, 4)\"\n    R.test(\"blur_score\", bl)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_13_quality as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Nesnel kalite** ölçütleri, insanların öznel değerlendirmesini (pahalı, yavaş) taklit etmeye çalışır. Referansın ne kadarının bilindiğine göre: **tam referanslı** (MSE, PSNR, SSIM), **indirgenmiş referanslı** (yalnız bazı öznitelikler), **referanssız** (yalnız bozuk görüntü).

**MSE / PSNR:** $\text{MSE}=\frac1N\sum(x_i-y_i)^2$, $\text{PSNR}=10\log_{10}\frac{R^2}{\text{MSE}}$. Basit, türevlenebilir, fiziksel anlamı açık; ama hata türüne kör: ortalama kayması, kontrast germe, bulanıklık ve JPEG aynı MSE'de çok farklı görünür.

**SSIM:** doğal görüntüler yapılıdır (komşu pikseller güçlü bağımlı) ve görme sistemi yapıyı çıkarmaya uyarlanmıştır. Benzerliği üçe ayır:

- parlaklık $l=\frac{2\mu_x\mu_y+C_1}{\mu_x^2+\mu_y^2+C_1}$,
- kontrast $c=\frac{2\sigma_x\sigma_y+C_2}{\sigma_x^2+\sigma_y^2+C_2}$,
- yapı $s=\frac{\sigma_{xy}+C_3}{\sigma_x\sigma_y+C_3}$,

$C_1=(K_1L)^2$, $C_2=(K_2L)^2$, $K_1=0.01$, $K_2=0.03$. $\alpha=\beta=\gamma=1$, $C_3=C_2/2$ ile

$$\text{SSIM}=\frac{(2\mu_x\mu_y+C_1)(2\sigma_{xy}+C_2)}{(\mu_x^2+\mu_y^2+C_1)(\sigma_x^2+\sigma_y^2+C_2)}.$$

Özellikler: simetri, $\le1$, ve yalnız $x=y$ iken 1. İstatistikler **yerel pencerede** (11 × 11 Gauss, $\sigma=1.5$) hesaplanır; SSIM haritasının ortalaması küresel SSIM'dir.

**MS-SSIM:** görüntüyü 5 ölçekte (her seferinde 2 × 2 ortalama + alt örnekleme) karşılaştır; kontrast–yapı terimleri her ölçekte, parlaklık yalnız en kabada: $\prod_jcs_j^{\beta_j}\cdot\text{SSIM}_M^{\beta_M}$.

## 1 · Elle

$l=\frac{2(2.5)(3.5)+6.5025}{2.5^2+3.5^2+6.5025}=\frac{24.0025}{25.0025}=0.96$. $y=x+1$ olduğu için $\sigma_x=\sigma_y=\sigma_{xy}$, kontrast–yapı çarpanı tam 1: SSIM yalnız parlaklık farkından düşer.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**MSE ve PSNR: $x=[10,20,30,40]$, $y=[12,18,30,44]$**

- `mse`: MSE
- `psnr`: PSNR (dB, 8 bit)

**SSIM (tek pencere, yansız σ, $K_1=0.01$, $K_2=0.03$, $L=255$): $x=[1,2,3,4]$, $y=[2,3,4,5]$**

- `c1`: $C_1=(K_1L)^2$
- `l`: Parlaklık $l(x,y)$
- `s`: Yapı $s(x,y)=\sigma_{xy}/(\sigma_x\sigma_y)$
- `ssim`: SSIM ($C_3=C_2/2$)
- `sneg`: $y=[4,3,2,1]$ için yapı terimi $s$

In [ ]:
cevaplar = dict(
    mse=None,
    psnr=None,
    c1=None,
    l=None,
    s=None,
    ssim=None,
    sneg=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"mse":[6,0],"psnr":[40.3493,0.001],"c1":[6.5025,0.0001],"l":[0.960004,0.0001],"s":[1,0],"ssim":[0.960004,0.0001],"sneg":[-1,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, düz listeler.

- `mse(x, y)`, `psnr(x, y, R=255)` (MSE = 0 ise `float("inf")`).
- `ssim_global(x, y, L=255, K1=0.01, K2=0.03)` — **tek pencere** (tüm liste); ortalamalar, **yansız** varyans ve kovaryans ($N-1$), birleşik formül.
- `ssim_components(x, y, L=255, K1=0.01, K2=0.03)` → `(l, c, s)`, $C_3=C_2/2$. Çarpımları `ssim_global`'a eşit olmalı.

In [ ]:
%%writefile stage_b.py
import math


def mse(x, y):
    ...


def psnr(x, y, R=255):
    ...


def ssim_global(x, y, L=255, K1=0.01, K2=0.03):
    ...


def ssim_components(x, y, L=255, K1=0.01, K2=0.03):
    ...


print(ssim_global([1, 2, 3, 4], [2, 3, 4, 5]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy (`scipy.ndimage` serbest), **döngü yok** (bisection gibi arama döngüleri hariç, aşağıya bak).

**`ssim(x, y, R=255)`** — `skimage.metrics.structural_similarity(x, y, data_range=R, gaussian_weights=True, sigma=1.5, use_sample_covariance=False)` ile **aynı**:

- `float64`; $\mu$, $E[x^2]$, $E[y^2]$, $E[xy]$ = `ndimage.gaussian_filter(…, sigma=1.5, truncate=3.5, mode="reflect")`.
- $v_x=E[x^2]-\mu_x^2$ (normalizasyon 1), SSIM haritası, kenardan `pad = 5` piksel kırpılmış ortalaması.

**Eşit MSE bozulmaları** (Wang 2004, Şekil 2) — her biri `float64` döndürür, hedef MSE'ye çok yakın:

- `mean_shift(img, mse)`, `contrast(img, mse)` ($\mu+a(x-\mu)$, $(a-1)^2\sigma^2=\text{MSE}$).
- `noise(img, mse, seed=0)` — Gauss gürültüsü, MSE'si tam hedefe ölçeklenmiş.
- `impulse(img, mse, seed=0)` — rastgele sırayla pikselleri 0 ya da 255 yap (yazı-tura), kümülatif hata hedefe ulaşana dek (döngüsüz: `cumsum` + `searchsorted`).
- `blur(img, mse)` — `cv2.GaussianBlur(…, (0, 0), σ)`; σ'yı ikiye bölmeyle bul (bir `for` döngüsü serbest).
- `jpeg(img, mse)` — `cv2.imencode(".jpg", …, [cv2.IMWRITE_JPEG_QUALITY, q])`; MSE'si hedefe en yakın kalite (`q` üzerinde bir döngü serbest).

In [ ]:
%%writefile stage_c.py
import numpy as np
import cv2
from scipy import ndimage


def ssim(x, y, R=255):
    ...


def mean_shift(img, mse):
    ...


def contrast(img, mse):
    ...


def noise(img, mse, seed=0):
    ...


def impulse(img, mse, seed=0):
    ...


def blur(img, mse):
    ...


def jpeg(img, mse):
    ...


from skimage import data
print(ssim(data.camera(), mean_shift(data.camera(), 210)))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `ssim_t(x, y, R=255.0)` — `x, y (H, W)` `float64` tensör, c'deki SSIM'in **türevlenebilir** hali. Gauss çekirdeği (11, σ = 1.5, toplam 1) ile ayrılabilir `conv2d`; skimage'ın `reflect` kuralı **simetrik** yansımadır (`dcba|abcd`): `F.pad(mode="reflect")` bunu değil `REFLECT_101`'i yapar, o yüzden kenarları `flip` + `cat` ile doldur. Kenardan 5 kırpılmış ortalama.
- `mse_sphere(ref, mse, sign, steps=200, lr=2.0, seed=0)` — Wang ve Simoncelli'nin "en büyük ayrışma" deneyi: $e$ rastgele başla, $\lVert e\rVert^2/N=\text{MSE}$'ye ölçekle; her adımda $\nabla_e\,\text{SSIM}(ref, ref+e)$, küreye **teğet** bileşeni ($g-\frac{g\cdot e}{e\cdot e}e$), RMS'ine bölünmüş adım `sign·lr`, sonra yine küreye izdüşür. `ref + e` döndür (kırpma yok).

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn.functional as F


def ssim_t(x, y, R=255.0):
    ...


def mse_sphere(ref, mse, sign, steps=200, lr=2.0, seed=0):
    ...


x = torch.rand(64, 64, dtype=torch.float64) * 255
print(ssim_t(x, x))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`ms_ssim(x, y, R=255.0, weights=(0.0448, 0.2856, 0.3001, 0.2363, 0.1333))` — `x, y (B, H, W)` `float64` tensör → `(B,)`; tüm yığın **aynı anda**:

1. Her ölçekte d'deki Gauss süzgeci (simetrik yansıma), $l$ ve $cs$ haritaları, kenardan 5 kırpılmış ortalamalar: $\overline{cs}_j$ ve $\overline{l\cdot cs}_j$.
2. Ölçekler arası `F.avg_pool2d(…, 2)`.
3. $\text{MS-SSIM}=\prod_{j=1}^{M-1}\mathrm{relu}(\overline{cs}_j)^{w_j}\cdot\mathrm{relu}(\overline{l\cdot cs}_M)^{w_M}$.

Ölçek döngüsü (5 tekrar) serbest; görüntü döngüsü yok. En küçük boyut $11\cdot2^4=176$'dan büyük olmalı.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def ms_ssim(x, y, R=255.0, weights=(0.0448, 0.2856, 0.3001, 0.2363, 0.1333)):
    ...


x = torch.rand(2, 192, 192, dtype=torch.float64) * 255
print(ms_ssim(x, x))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · SSIM seçenekleri: düzgün pencere, örnek kovaryansı, tam harita

`ssim_map(x, y, win=7, sample_cov=True, R=255)` → `(mssim, S)`: `structural_similarity(x, y, win_size=win, use_sample_covariance=sample_cov, data_range=R, full=True)` ile aynı. Düzgün pencere (`ndimage.uniform_filter`, `mode="reflect"`), örnek kovaryansında normalizasyon $\frac{N}{N-1}$ ($N=win^2$), harita tam boyutlu, ortalama `pad = (win−1)/2` kırpılarak.

In [ ]:
%%writefile extra_x1.py
import numpy as np
from scipy import ndimage


def ssim_map(x, y, win=7, sample_cov=True, R=255):
    ...


from skimage import data
print(ssim_map(data.camera(), data.camera())[0])

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Renkli görüntüde PSNR ve SSIM: Y kanalı ve kenar kırpma

Süper çözünürlük makalelerinin yaygın protokolü:

- `rgb_to_y(img)` — `uint8` RGB → $Y=16+\frac{65.481R+128.553G+24.966B}{255}$ (`float64`, BT.601 "stüdyo" aralığı, MATLAB `rgb2ycbcr`).
- `psnr_y(a, b, shave=0)` — Y kanalında, kenardan `shave` piksel kırpılmış PSNR ($R=255$).
- `ssim_rgb(a, b)` — kanal başına SSIM'in ortalaması (`channel_axis` davranışı), c'deki Gauss tanımıyla.

In [ ]:
%%writefile extra_x2.py
import numpy as np
from scipy import ndimage


def rgb_to_y(img):
    ...


def psnr_y(a, b, shave=0):
    ...


def ssim_rgb(a, b):
    ...


from skimage import data
print(rgb_to_y(data.astronaut()).min())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Referanssız ölçütler: gürültü ve bulanıklık kestirimi

Referanssız ölçütler:

- `estimate_noise(img)` — Immerkær (1996): $\hat\sigma=\sqrt{\frac\pi2}\frac1{6(W-2)(H-2)}\sum|I*N|$, $N=\begin{bmatrix}1&-2&1\\-2&4&-2\\1&-2&1\end{bmatrix}$ (yalnız iç pikseller, `valid`).
- `blur_score(img)` — Laplacian varyansı: `cv2.Laplacian(img, cv2.CV_64F).var()`.

In [ ]:
%%writefile extra_x3.py
import math
import numpy as np
import cv2
from scipy import signal


def estimate_noise(img):
    ...


def blur_score(img):
    ...


from skimage import data
print(estimate_noise(data.camera()))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: SSIM makalesini yeniden üret

[Wang, Bovik, Sheikh, Simoncelli — Image Quality Assessment: From Error Visibility to Structural Similarity (IEEE TIP 2004)](https://doi.org/10.1109/TIP.2003.819861)

Makaledeki üç ana deneyi yeniden üret: (1) eşit MSE'li bozulmalar figürü (bu modülün c aşaması), (2) SSIM haritaları (JPEG ve gürültü için), (3) LIVE (ya da TID2013/KADID-10k) veritabanında öznel puanlarla (MOS/DMOS) korelasyon: PSNR, SSIM ve MS-SSIM için Pearson (lojistik regresyon sonrası), Spearman (SROCC) ve aykırılık oranı.


**Adımlar**

1. Veritabanını indir; referans–bozuk çiftleri ve DMOS değerlerini oku.
2. Makaledeki 5 parametreli lojistik fonksiyonu SciPy `curve_fit` ile uydur, ardından Pearson ve RMSE.
3. Pencere boyutu (8 × 8 düzgün vs 11 × 11 Gauss) ve alt örnekleme (makale: görüntü ≈ 256 piksele indirilir) etkisini ablasyonla göster.
4. Bu modülün d aşamasındaki 'MSE küresinde en iyi / en kötü SSIM' görüntülerini makaledeki tartışmayla birlikte yorumla.

**Kabul ölçütleri**

- Korelasyon tablosu: SSIM'in SROCC'si PSNR'ınkinden belirgin yüksek (LIVE'da ~0.95'e karşı ~0.88).
- Saçılım grafikleri (öznel puan – nesnel skor) ve uydurulan eğriler.
- Alt örneklemenin SROCC'ye etkisi raporlanmış.

## Makale ödevi: LPIPS: derin özniteliklerle algısal benzerlik (uzantı)

[Zhang, Isola, Efros, Shechtman, Wang — The Unreasonable Effectiveness of Deep Features as a Perceptual Metric (CVPR 2018)](https://arxiv.org/abs/1801.03924)

Önceden eğitilmiş bir CNN'in (AlexNet ya da VGG) katman özniteliklerini kanal bazında normalize edip doğrusal ağırlıklarla birleştiren LPIPS'i uygula; BAPPS 2AFC verisinde ağırlıkları eğit ve SSIM ile karşılaştır.


**Adımlar**

1. Öznitelik çıkarımı, birim normalize, kanal ağırlıkları $w_l$, uzamsal ortalama.
2. 2AFC kaybı: hangi bozulmuş yamanın referansa daha yakın olduğu (insan oyu).
3. SSIM, MS-SSIM, PSNR ve LPIPS için 2AFC doğruluğu tablosu.

**Kabul ölçütleri**

- BAPPS doğrulamada LPIPS 2AFC skoru SSIM'den en az 5 puan yüksek.

## Kaynaklar

- [Wang, Bovik, Sheikh, Simoncelli — Image Quality Assessment: From Error Visibility to Structural Similarity (IEEE TIP 2004)](https://doi.org/10.1109/TIP.2003.819861) — ©
- [Wang, Simoncelli, Bovik — Multiscale structural similarity for image quality assessment (Asilomar 2003)](https://doi.org/10.1109/ACSSC.2003.1292216) — ©
- [scikit-image — metrics.structural_similarity](https://scikit-image.org/docs/stable/api/skimage.metrics.html) — BSD-3
- [Immerkær — Fast Noise Variance Estimation (CVIU 1996)](https://doi.org/10.1006/cviu.1996.0060) — ©